# 🔧 Notebook 2: Read-Repair on the Fly

**Read-repair** = during a read, the coordinator queries multiple replicas, picks the freshest answer, **and pushes that answer back to any replica that returned a stale value**. Convergence happens lazily, paid for by reads (not by background scans).


## 🛠️ Setup

```bash
cd 02-distributed-primitives/read-repair
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟩 Implementation

In [ ]:
from dataclasses import dataclass, field
from typing import Dict, Tuple, List

@dataclass
class Replica:
    name: str
    data: Dict[str, Tuple[str,int]] = field(default_factory=dict)
    def write(self, k, v, ts):
        cur = self.data.get(k)
        if cur is None or ts > cur[1]:
            self.data[k] = (v, ts)
    def read(self, k): return self.data.get(k)

class Coordinator:
    def __init__(self, replicas: List[Replica]):
        self.replicas = replicas

    def read(self, key, R=2):
        # Query R replicas (here we just take the first R for demo simplicity)
        responses = [(r, r.read(key)) for r in self.replicas[:R]]
        # Pick the one with the highest timestamp
        valid = [(r, val) for r, val in responses if val is not None]
        if not valid: return None
        winner_replica, (winner_val, winner_ts) = max(valid, key=lambda x: x[1][1])
        # Repair every replica with a stale (or missing) value — even those we didn't query
        for r in self.replicas:
            cur = r.read(key)
            if cur is None or cur[1] < winner_ts:
                print(f'  🔧 repairing {r.name}: {cur} -> ({winner_val!r}, {winner_ts})')
                r.write(key, winner_val, winner_ts)
        return winner_val, winner_ts

r1, r2, r3 = Replica('r1'), Replica('r2'), Replica('r3')
r1.write('user:42','Alice v2',200)
r2.write('user:42','Alice v2',200)
r3.write('user:42','Alice v1',100)

coord = Coordinator([r1,r2,r3])
print('read 1:', coord.read('user:42'))
print('read 2:', coord.read('user:42'))  # nothing left to repair
print('r3 now has:', r3.read('user:42'))


## 📊 Read-repair vs anti-entropy repair

| | Read-repair | Anti-entropy (Merkle scan) |
|---|---|---|
| When it happens | every read | scheduled / on-demand |
| Cost paid by | client read latency (slightly) | background CPU + network |
| Coverage | only **hot** keys | every key, eventually |
| Fixes a key after... | it is read | the next sweep |

Real systems (Cassandra, DynamoDB) use **both** — read-repair keeps hot data fresh; anti-entropy catches the long tail of cold keys.